In [ ]:

import numpy as np
import networkx as nx
from scipy.stats import qmc
import EoN
from pathlib import Path
import pickle
from scipy.stats import qmc
from pathlib import Path
from tqdm import tqdm
from pathlib import Path
import csv
import time
import datetime
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("experiments/mcmc-sampling/data/raw")
PLOTS_DIR = Path("experiments/mcmc-sampling/out/plots/mcmc_sampling_plots")
MCMC_DIR = Path("experiments/mcmc-sampling/data/raw/mcmc_posterior_results")
RESULTS_DIR = Path("experiments/mcmc-sampling/out/results")
OUTPUT_PKL = DATA_DIR / "abm-data.pkl"
OUTPUT_CSV = DATA_DIR / "abm-data.csv"
TIMING_FILE = RESULTS_DIR / "timing.txt"


# PARAMETERS
N=100000              # network size
m=10                      # Barabasi–Albert attachment parameter

tmax=250
n_timepoints=250
initial_samples=10000 
sigma = 0.5    # width of R0 target distribution
n_replicates=1  # replicates of parameter sets 

PARAM_RANGES = {  
    'tau':(0.0003,0.02),
    'gamma':(0.03,1),
    'rho':(0.0001,0.01)

}



PARAM_NAMES = ['tau', 'gamma', 'rho']
output_path = Path('abm-data.pkl')

# results_k_avg = []
# results_k2_avg = []
# results_ratio = []

# for i in range(1000):
    
#     G = nx.barabasi_albert_graph(N, m)
    
#     degrees = np.array([d for _, d in G.degree()])
    
#     k_avg = degrees.mean()
#     k2_avg = (degrees**2).mean()
#     ratio = (k2_avg-k_avg)/ k_avg   # R0 = (tau/gamma) * (<k²> - <k>) / <k> that is our BA network ratio for R0 computation 
    
#     results_k_avg.append(k_avg)
#     results_k2_avg.append(k2_avg)
#     results_ratio.append(ratio)

# # converting to arrays
# results_k_avg = np.array(results_k_avg)
# results_k2_avg = np.array(results_k2_avg)
# results_ratio = np.array(results_ratio)

# # means
# mean_k_avg = results_k_avg.mean()
# mean_k2_avg = results_k2_avg.mean()
# mean_ratio = results_ratio.mean()

# # standard errors 
# se_k_avg = results_k_avg.std(ddof=1)/np.sqrt(len(results_k_avg))
# se_k2_avg = results_k2_avg.std(ddof=1)/np.sqrt(len(results_k2_avg))
# se_ratio = results_ratio.std(ddof=1)/np.sqrt(len(results_ratio))

# # 95% CI  (mean ± 1.96 * standard error)
# ci_k_avg = (mean_k_avg - 1.96*se_k_avg, mean_k_avg + 1.96*se_k_avg)
# ci_k2_avg = (mean_k2_avg - 1.96*se_k2_avg, mean_k2_avg + 1.96*se_k2_avg)
# ci_ratio = (mean_ratio - 1.96*se_ratio, mean_ratio + 1.96*se_ratio)

# print("Mean <k>:", mean_k_avg, "CI:", ci_k_avg)
# print("Mean <k^2>:", mean_k2_avg, "CI:", ci_k2_avg)
# print("Mean <k^2>/<k>:", mean_ratio, "CI:", ci_ratio)






#ratio=58.979# calculated for 1000 graphs
ratio = 58# calculated for 1000 graphs
net_stats = {
    'k_avg': 19.99,
    'k2_avg': 1199.47,
    'ratio': ratio,
    
}



# R0 COMPUTATION
def compute_R0(samples,ratio):
    """
    Epidemic reproduction number.

    R0 = (tau/gamma) * (<k²> - <k>) / <k> 
    """

    tau = samples[:, 0]
    gamma = samples[:, 1]

    R0 = (tau/gamma) * ratio  

    return R0

#PRIOR SPECIFICATION
# Using MCMC-NUTS to sample from the target distribution  with Uniform distribution

t0_data_gen = time.perf_counter()

with pm.Model() as model:

    # Priors: Uniform over plausible ranges
    tau= pm.Uniform("tau", lower=PARAM_RANGES['tau'][0], upper=PARAM_RANGES['tau'][1])
    gamma= pm.Uniform("gamma", lower=PARAM_RANGES['gamma'][0], upper=PARAM_RANGES['gamma'][1])
    rho = pm.Uniform("rho", lower=PARAM_RANGES['rho'][0], upper=PARAM_RANGES['rho'][1])

    # Computing R0 deterministically
    R0 = pm.Deterministic("R0", (tau/gamma) * ratio)

    # Target density: Gaussian near the epidemic threshold (R0=1) with width sigma that controls the concentration of samples around R0=1
    logp = -0.5 * ((R0 - 1.0) / sigma) ** 2
    pm.Potential("R0_target", logp)

    # Sampling with NUTS
    trace = pm.sample(
        draws=initial_samples,
        tune=3000,
        chains=4 ,  # thus Total sample=samples*chains
        cores=1,
        thin=10,
        target_accept=0.95,
        random_seed=43
    )
#Thinning after sampling.
trace_thinned = trace.sel(draw=slice(None, None, 10))

data_gen_time = time.perf_counter() - t0_data_gen
print(f"Data generation time (uniform MCMC): {data_gen_time:.2f}s")

# Converting trace_thinned to (n_samples, 3) array: tau, gamma, rho
posterior_samples=np.vstack([
    trace_thinned.posterior['tau'].values.flatten(),
    trace_thinned.posterior['gamma'].values.flatten(),
    trace_thinned.posterior['rho'].values.flatten()
]).T

# Saving posterior samples to CSV
summary_df = az.summary(trace_thinned, var_names=["tau", "gamma", "rho", "R0"], hdi_prob=0.95)
print(summary_df)
summary_df.to_csv(MCMC_DIR/ "mcmc_summary.csv")

r0_samples = trace_thinned.posterior["R0"].values.flatten()
r0_lo, r0_hi = np.percentile(r0_samples, [5, 95])
print(f"\n95% posterior interval for R0: [{r0_lo:.4f}, {r0_hi:.4f}]")
print(f"Median R0: {np.median(r0_samples):.4f}  :  Mean R0: {r0_samples.mean():.4f}")

#Plotting trace plots for tau, gamma, rho
az.plot_trace(
    trace,
    var_names=["tau", "gamma", "rho"],
    compact=False,
    figsize=(12, 8)
)

plt.tight_layout()
trace_plot_path = PLOTS_DIR / "mcmc_trace_plot.png"
plt.savefig(trace_plot_path, dpi=600, bbox_inches='tight')
plt.show()
print(f"Saved: {trace_plot_path}")


#posterior sample dimensions 
print(posterior_samples.shape) #4000,3

# Function to run batch simulations
def run_batch(G, posterior_samples, n_replicates, tmax, n_timepoints, seed=None):
    
    if seed is not None:
        np.random.seed(seed)

    t_fixed = np.linspace(0, tmax, n_timepoints)
    all_sims = []

    for tau, gamma, rho in tqdm(posterior_samples, desc="Running batch simulations"):

        for rep in range(n_replicates):

            t, S, I, R = EoN.fast_SIR(G, tau, gamma, rho=rho, tmax=tmax)

            all_sims.append({
                'params': {
                    'tau': float(tau),
                    'gamma': float(gamma),
                    'rho': float(rho),
                },
                'output': {
                    't': t_fixed,
                    'S': np.interp(t_fixed, t, S),
                    'I': np.interp(t_fixed, t, I),
                    'R': np.interp(t_fixed, t, R),
                },
                'replicate_id': rep,
            })

    print(f"Generated {len(all_sims)} simulations "
          f"({len(posterior_samples)} param sets × {n_replicates} replicates)")

    return all_sims



# Building dataset structure
def build_dataset(all_sims, G, net_stats, m, n_replicates, param_ranges):

    n_timepoints = len(all_sims[0]['output']['t'])

    dataset = {

        'simulations': all_sims,

        'network': {
            'type': 'barabasi_albert',
            'N': G.number_of_nodes(),
            'm': m,
            'ratio': net_stats['ratio'],
            'graph': G,
        },

        'metadata': {
            'n_samples': len(all_sims),
            'n_replicates': n_replicates,
            'n_timepoints': n_timepoints,
            'param_ranges': param_ranges,
            'R0_formula': 'R0 = (tau/gamma) * <k²>/<k>',
            'sampling_strategy': 'MCMC',
        }
    }

    return dataset

# Aggregating simulations for plotting
def summarise_for_plot(dataset):

    sims = dataset['simulations']
    ratio = dataset['network']['ratio']
    t_fixed = sims[0]['output']['t']

    all_I = np.array([s['output']['I'] for s in sims])
    all_S = np.array([s['output']['S'] for s in sims])
    all_R = np.array([s['output']['R'] for s in sims])

    R0_vals = np.array([(s['params']['tau'] / s['params']['gamma']) * ratio for s in sims])

    return {
        't': t_fixed,
        'all_I': all_I,
        'all_S': all_S,
        'all_R': all_R,
        'R0': R0_vals,
        'I_mean': all_I.mean(axis=0),
        'I_p10': np.percentile(all_I, 10, axis=0),
        'I_p25': np.percentile(all_I, 25, axis=0),
        'I_p75': np.percentile(all_I, 75, axis=0),
        'I_p90': np.percentile(all_I, 90, axis=0),
    }


# Plot for epidemic uncertainty — three R0 regimes
def plot_sir_uncertainty(full_results, output_dir=PLOTS_DIR):

    t = full_results['t']
    all_I = full_results['all_I']
    R0_vals = full_results['R0']
    total = len(R0_vals)

    sub_mask  = R0_vals < 0.1
    near_mask = (R0_vals >= 0.1) & (R0_vals <= 2)
    out_mask  = R0_vals > 2

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    fig.suptitle(
        "Epidemic Trajectories by R₀ Regime",
        fontsize=13,
        fontweight='bold'
    )

    for ax, mask, label, color in [
        (axes[0], sub_mask,  "Sub-threshold (R₀ < 0.1)",          "steelblue"),
        (axes[1], near_mask, "Near-threshold (0.1 ≤ R₀ ≤ 2)", "goldenrod"),
        (axes[2], out_mask,  "Outbreak (R₀ > 2)",                  "firebrick"),
    ]:
        subset = all_I[mask]
        n = mask.sum()

        if n == 0:
            ax.set_title(f"{label}\n(no trajectories)")
            ax.grid(True, alpha=0.3)
            continue

        mean_I = subset.mean(axis=0)
        p10 = np.percentile(subset, 10, axis=0)
        p25 = np.percentile(subset, 25, axis=0)
        p75 = np.percentile(subset, 75, axis=0)
        p90 = np.percentile(subset, 90, axis=0)

        ax.fill_between(t, p10, p90, color=color, alpha=0.15, label='10–90th pct')
        ax.fill_between(t, p25, p75, color=color, alpha=0.30, label='25–75th pct')
        ax.plot(t, mean_I, color=color, linewidth=2, label='Mean')

        ax.set_xlabel("Time")
        ax.set_ylabel("Number infectious")
        ax.set_title(f"{label}\n(n={n}, {n/total*100:.1f}%)")
        ax.set_ylim(bottom=0)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    output_dir = Path(output_dir)
    out_path   = output_dir / "trajectories_split.png"
    plt.tight_layout()
    plt.savefig(out_path, dpi=600, bbox_inches='tight')
    plt.show()
    print(f"Saved: {out_path}")



# Save dataset (pickle)
def save_dataset(dataset, filepath):

    with open(filepath, 'wb') as f:
        pickle.dump(dataset, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Dataset saved {filepath} ({len(dataset['simulations'])} sims)")



# Save summary CSV
def save_csv(dataset, filepath):
    sims = dataset['simulations']
    ratio = dataset['network']['ratio']
    N = dataset['network']['N']

    fields = [
        'sim_id', 'replicate_id',
        'tau', 'gamma', 'rho',
        'R0', 'peak_I', 'peak_time',
        'final_R', 'attack_rate', 'near_threshold'
    ]

    with open(filepath, 'w', newline='') as f:

        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()

        for sim_id, sim in enumerate(sims):

            tau = sim['params']['tau']
            gamma = sim['params']['gamma']
            rho = sim['params']['rho']

            R0 = (tau / gamma) * ratio

            I = sim['output']['I']
            R = sim['output']['R']
            t = sim['output']['t']

            peak_I = float(I.max())

            peak_time = float(t[I.argmax()]) if peak_I > 0 else np.nan

            writer.writerow({

                'sim_id': sim_id,
                'replicate_id': sim.get('replicate_id', 0),

                'tau': tau,
                'gamma': gamma,
                'rho': rho,

                'R0': round(R0, 4),

                'peak_I': peak_I,
                'peak_time': peak_time,

                'final_R': float(R[-1]),
                'attack_rate': float(R[-1] / N),

                'near_threshold': int(0.1 <= R0 <= 2),
            })

    print(f"CSV saved to {filepath} ({len(sims)} rows)")

# Entry point
G = nx.barabasi_albert_graph(N, m)

t0_sim = time.perf_counter()
all_sims = run_batch(
    G,
    posterior_samples,
    n_replicates,
    tmax,
    n_timepoints,
)
sim_time = time.perf_counter() - t0_sim
print(f"Simulation time: {sim_time:.2f}s")

dataset = build_dataset(
    all_sims,
    G,
    net_stats,
    m,
    n_replicates,
    PARAM_RANGES,
)

full_results = summarise_for_plot(dataset)

# Plots
plot_sir_uncertainty(full_results, output_dir=PLOTS_DIR)

# Data
save_dataset(dataset, OUTPUT_PKL)   
save_csv(dataset, OUTPUT_CSV)       

# Exploration 
data = pd.read_csv(OUTPUT_CSV)      

print(data.columns)
print(data.head(20))
print(f"\nTotal rows   : {len(data)}")
print(f"Missing values:\n{data.isnull().sum()}")
print(f"Shape: {data.shape}")
print(f"\n{data.describe(include='all')}")

summary = data.describe(include='all')

summary.to_csv(DATA_DIR / "summary.csv")

total_samples = len(data)
greater= data[data['R0'] > 2]
between = data[(data['R0'] >= 0.1) & (data['R0'] <= 2)]
less_than = data[data['R0'] < 0.1]

print(f"\nR0 > 2      : {len(greater)}  ({len(greater)/total_samples*100:.1f}%)")
print(f"0.1 ≤ R0 ≤ 2: {len(between)} ({len(between)/total_samples*100:.1f}%)")
print(f"R0 < 0.1    : {len(less_than)}  ({len(less_than)/total_samples*100:.1f}%)")

print(f"\nData : {DATA_DIR}")
print(f"Plots : {PLOTS_DIR}")

# Write timing report
n_samples = len(posterior_samples)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(TIMING_FILE, "w") as f:
    f.write("=" * 50 + "\n")
    f.write("MCMC Sampling — Timing Report\n")
    f.write(f"Run date : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Samples generated : {n_samples}\n")
    f.write(f"Replicates        : {n_replicates}\n")
    f.write(f"Network size (N)  : {N}\n\n")
    f.write(f"Data generation time (uniform MCMC) : {data_gen_time:.2f} s  ({data_gen_time/60:.2f} min)\n")
    f.write(f"Simulation time                     : {sim_time:.2f} s  ({sim_time/60:.2f} min)\n")
    f.write(f"Total time                          : {(data_gen_time + sim_time):.2f} s  ({(data_gen_time + sim_time)/60:.2f} min)\n")
print(f"Timing saved: {TIMING_FILE}")


#Saves to PLOTS_DIR
# R0 = (tau/gamma) * ratio  =>  tau = R0 * gamma / ratio
slope = 1 / ratio  # R0=1 line slope
plt.figure(figsize=(10, 10))
plt.scatter(data['gamma'], data['tau'], alpha=0.3, s=10, color='steelblue', label='MCMC samples')

x_vals = np.linspace(data['gamma'].min(), data['gamma'].max(), 300)

# Near-threshold shaded region: 0.1 <= R0 <= 2.0
y_lo = (0.1 / ratio) * x_vals
y_hi = (2.0 / ratio) * x_vals
plt.fill_between(x_vals, y_lo, y_hi,
                 alpha=0.20, color='gold', label='Near threshold (0.1 ≤ R₀ ≤ 2.0)')
plt.plot(x_vals, y_lo, color='goldenrod', linestyle=':', linewidth=1.5, label='R₀ = 0.1')
plt.plot(x_vals, y_hi, color='darkorange', linestyle=':', linewidth=1.5, label='R₀ = 2.0')

# Epidemic threshold: R0 = 1
y_r0_1 = slope * x_vals
plt.plot(x_vals, y_r0_1, color='red', linestyle='--', linewidth=2, label='R₀ = 1 (epidemic threshold)')

tau_max = data['tau'].max() * 1.1
plt.ylim(0, tau_max)
plt.xlim(data['gamma'].min() * 0.95, data['gamma'].max() * 1.02)

plt.xlabel('gamma', fontsize=13)
plt.ylabel('tau', fontsize=13)
plt.title('Scatter plot of tau vs gamma — MCMC Posterior Samples', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

scatter_path = PLOTS_DIR / "tau_gamma_scatter.png"
plt.savefig(scatter_path, dpi=600, bbox_inches='tight')
plt.show()
print(f"Saved: {scatter_path}")



data = data

params = {
    'R0':('R0', 'R\u2080','steelblue'),
    'tau':('tau','\u03C4 (transmission rate)', 'darkorange'),
    'rho':('rho','\u03C1 (initial infected)',  'mediumpurple'),
    'gamma':('gamma','\u03C1 (recovery rate)',  'red'),
    'attack_rate': ('attack_rate', '(Attack Rate)','firebrick'),
    'final_R':('final_R','Final Recovered (R)','seagreen'),
    'near_threshold': ('near_threshold', 'Near Threshold (R\u2080\u22481)', 'goldenrod'),
    'peak_I': ('peak_I','(peak of infected compartment)','green'),
}

fig, axes = plt.subplots(2, 4, figsize=(16, 9))
axes = axes.flatten()

for ax, (col, (key, label, color)) in zip(axes, params.items()):
    values = data[key].dropna()

    if key == 'near_threshold':
        counts = values.value_counts().sort_index()
        ax.bar(counts.index.astype(str), counts.values, color=[color, 'lightgray'][:len(counts)],
               alpha=0.85, edgecolor='white', linewidth=0.5)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Not near (0)', 'Near (1)'], fontsize=10)
        pct = values.mean() * 100
        ax.set_title(f'{label}\n({pct:.1f}% near threshold)', fontsize=11, fontweight='bold')
    else:
        counts, bin_edges = np.histogram(values, bins=30)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        width = bin_edges[1] - bin_edges[0]
        ax.bar(bin_centers, counts, width=width * 0.85, color=color, alpha=0.85,
               edgecolor='white', linewidth=0.4)
        ax.axvline(values.mean(), color='black', linestyle='--', linewidth=1.5,
                   label=f'Mean = {values.mean():.4f}')
        ax.legend(fontsize=9)
        ax.set_title(f'Distribution of {label}', fontsize=11, fontweight='bold')

    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('MCMC Posterior — Parameter & Outcome Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()

out = PLOTS_DIR / "parameter_bar_plots.png"
plt.savefig(out, dpi=600, bbox_inches='tight')
plt.show()
print(f"Saved: {out}")


WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
c:\Users\gmm71\OneDrive - University of Cambridge\Documents\Emulation models\abm-epidemic-emulator\venv\lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (4 chains in 1 job)
NUTS: [tau, gamma, rho]


Sampling 4 chains for 3_000 tune and 10_000 draw iterations (12_000 + 40_000 draws total) took 198 seconds.


Data generation time (uniform MCMC): 200.86s
        mean     sd  hdi_2.5%  hdi_97.5%  mcse_mean  mcse_sd  ess_bulk  \
tau    0.010  0.005     0.001      0.019      0.000    0.000    4000.0   
gamma  0.633  0.231     0.216      0.999      0.004    0.002    3908.0   
rho    0.005  0.003     0.001      0.010      0.000    0.000    4080.0   
R0     0.936  0.408     0.087      1.652      0.006    0.004    4089.0   

       ess_tail  r_hat  
tau      4055.0    1.0  
gamma    3786.0    1.0  
rho      3853.0    1.0  
R0       4105.0    1.0  


OSError: Cannot save file into a non-existent directory: 'experiments\mcmc-sampling\data\raw\mcmc_posterior_results'

: 

In [ ]:
"""
adaptive_sampling_mcmc.py
─────────────────────────
Physics-informed MCMC-NUTS sampling for the SIR epidemic emulator.

Key design decisions
────────────────────
1.  R₀ is sampled DIRECTLY from TruncatedNormal(μ=1, σ=0.7, lo=0.05, hi=10).
    This guarantees a near-normal R₀ distribution centred at the epidemic
    threshold, rather than deriving R₀ from uniform τ/γ (which produces a
    right-skewed R₀ prior that dominates a wide Gaussian potential).

2.  τ is derived deterministically as τ = R₀ × γ / ratio.
    A hard bound potential rejects (R₀, γ) pairs producing τ outside
    [τ_min, τ_max], preserving physical validity.

3.  σ = 0.7  (not 3).  With σ=3 the Gaussian potential is essentially
    flat across R₀ ∈ [0, 5] and the sampler reverts to the prior,
    producing the observed right-skewed distribution.

4.  Thinning is applied ONCE (post-hoc only).  The earlier script had
    thin=10 inside pm.sample AND a second .sel(slice(None,None,10)),
    discarding 99% of draws.
"""

import numpy as np
import networkx as nx
import pymc as pm
import arviz as az
import EoN
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import csv
import time
import datetime
import warnings
from pathlib import Path
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ── Directories ───────────────────────────────────────────────────────────
DATA_DIR    = Path("experiments/mcmc-sampling/data/raw")
PLOTS_DIR   = Path("experiments/mcmc-sampling/out/plots/mcmc_sampling_plots")
MCMC_DIR    = Path("experiments/mcmc-sampling/data/raw/mcmc_posterior_results")
RESULTS_DIR = Path("experiments/mcmc-sampling/out/results")
OUTPUT_PKL  = DATA_DIR / "abm-data.pkl"
OUTPUT_CSV  = DATA_DIR  / "abm-data.csv"
TIMING_FILE = RESULTS_DIR / "timing.txt"

for d in [DATA_DIR, PLOTS_DIR, MCMC_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Network parameters
N = 100000   # population size
m = 10        # BA attachment parameter
ratio= 58        # (⟨k²⟩ − ⟨k⟩)/⟨k⟩  — estimated from 1000 BA graphs

net_stats = {
    'k_avg' : 19.99,
    'k2_avg': 1199.47,
    'ratio'  : ratio,
}

# ── Simulation parameters 
tmax= 250
n_timepoints  = 250
n_replicates  = 1

# MCMC parameters 
N_DRAWS = 10000   # posterior draws to keep (= training set size)
N_TUNE = 3000    # NUTS warm-up steps
N_CHAINS = 4        # parallel chains
THIN_FACTOR = 10       # post-hoc thinning  (applied ONCE, not twice)
SIGMA_R0 = 3     # concentration width — CRITICAL: 0.7 not 3

# Parameter bounds 
PARAM_RANGES = {
    'tau'  : (0.0003, 0.02),
    'gamma': (0.03,   1.0),
    'rho'  : (0.0001, 0.01),
    'R0'   : (0.05,   10.0),
}



#  1.  MCMC-NUTS SAMPLING  


def run_mcmc():
    """
    Sample (R₀, γ, ρ) using NUTS then derive τ = R₀ × γ / ratio.

    Model
    ─────
    γ   ~ Uniform(γ_min, γ_max)
    ρ   ~ Uniform(ρ_min, ρ_max)
    R₀  ~ TruncatedNormal(μ=1, σ=0.7, lower=0.05, upper=10)
    τ   = R₀ × γ / ratio          [deterministic]
    constraint: τ ∈ [τ_min, τ_max]  [hard potential]

    Why reparametrise?
    ──────────────────
    Sampling τ and γ from independent Uniform priors and computing
    R₀ = τ/γ × ratio produces a right-skewed R₀ distribution (ratio
    of two uniforms).  A wide Gaussian potential (σ=3) barely reshapes
    this, leaving R₀ concentrated in [0.5, 2] rather than near 1.
    Sampling R₀ directly from TruncatedNormal guarantees the intended
    near-normal shape centred at the epidemic threshold.

    Returns
    -------
    posterior_samples : (n_samples, 3) array  [tau, gamma, rho]
    trace_thinned     : ArviZ InferenceData object
    """
    print(f"\nMCMC-NUTS sampling  (σ_R₀ = {SIGMA_R0}, "
          f"{N_DRAWS} draws × {N_CHAINS} chains)")

    with pm.Model() as model:

        # Prior on R0
        R0 = pm.TruncatedNormal(
            "R0",
            mu=1.0,
            sigma=2,
            lower=0.05,
            upper=10.0,
        )

        # Prior on tau
        tau = pm.Uniform(
            "tau",
            lower=0.0003,
            upper=0.02,
        )

        # Compute gamma
        gamma = pm.Deterministic(
            "gamma",
            tau * ratio / R0
        )

        # rho
        rho = pm.Uniform(
            "rho",
            lower=0.0001,
            upper=0.01,
        )

        # Soft penalty for gamma outside desired range
        pm.Potential(
            "gamma_penalty",
            pm.math.switch(
                (gamma >= 0.03) &
                (gamma <= 1.0),
                0,
                -20,
            ),
        )

        trace = pm.sample(
            draws=N_DRAWS,
            tune=3000,
            target_accept=0.8,
            random_seed=43,
        )

    # Post-hoc thinning: keep every THIN_FACTOR-th draw 
    trace_thinned = trace.sel(draw=slice(None, None, THIN_FACTOR))

    # Diagnostics 
    summary_df = az.summary(trace_thinned,
                            var_names=["R0", "tau", "gamma", "rho"],
                            hdi_prob=0.95)
    print("\nPosterior summary:")
    print(summary_df)
    summary_df.to_csv(MCMC_DIR / "mcmc_summary.csv")

    r0_vals = trace_thinned.posterior["R0"].values.flatten()
    print(f"\nR₀  mean  : {r0_vals.mean():.4f}")
    print(f"R₀  median: {np.median(r0_vals):.4f}")
    print(f"R₀  std   : {r0_vals.std():.4f}")
    print(f"R₀  5–95% : [{np.percentile(r0_vals,5):.4f}, "
          f"{np.percentile(r0_vals,95):.4f}]")

    #  Trace plots 
    az.plot_trace(trace, var_names=["R0", "tau", "gamma", "rho"],
                  compact=False, figsize=(12, 10))
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "mcmc_trace_plot.png", dpi=600,
                bbox_inches="tight")
    plt.close()
    print(f"Saved: {PLOTS_DIR / 'mcmc_trace_plot.png'}")

    # Extract (tau, gamma, rho) as (n_samples, 3) array 
    posterior_samples = np.vstack([
        trace_thinned.posterior["tau"].values.flatten(),
        trace_thinned.posterior["gamma"].values.flatten(),
        trace_thinned.posterior["rho"].values.flatten(),
    ]).T                                           # (n_samples, 3)

    print(f"\nPosterior samples shape: {posterior_samples.shape}")
    return posterior_samples, trace_thinned


# ══════════════════════════════════════════════════════════════════════════
#  2.  R₀ DISTRIBUTION DIAGNOSTIC PLOT
# ══════════════════════════════════════════════════════════════════════════

def plot_r0_distribution(posterior_samples, out_dir=PLOTS_DIR):
    """
    Histogram of sampled R₀ values with threshold annotations.
    Should be approximately normal centred at 1.
    """
    tau   = posterior_samples[:, 0]
    gamma = posterior_samples[:, 1]
    R0    = (tau / gamma) * ratio

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # ── Panel (a): R₀ histogram ───────────────────────────────────────────
    ax = axes[0]
    ax.hist(R0, bins=60, color="blue", alpha=0.75, edgecolor="white")
    ax.axvline(1.0, color="red",      lw=2.0, linestyle="--",
               label=r"$\mathcal{R}_0 = 1$ (threshold)")
    ax.axvline(R0.mean(), color="black", lw=1.5, linestyle=":",
               label=f"Mean = {R0.mean():.3f}")
    ax.set_xlabel(r"$\mathcal{R}_0$", fontsize=12)
    ax.set_ylabel("Count",            fontsize=12)
    ax.set_title(r"(a)  Sampled $\mathcal{R}_0$ distribution"
                 f"\nσ = {SIGMA_R0}  (TruncatedNormal)", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.spines[["top", "right"]].set_visible(False)

    # ── Panel (b): τ vs γ scatter coloured by R₀ ─────────────────────────
    ax = axes[1]
    sc = ax.scatter(gamma, tau, c=R0, cmap="RdYlGn_r",
                    s=6, alpha=0.5, vmin=0, vmax=4)
    plt.colorbar(sc, ax=ax, label=r"$\mathcal{R}_0$")

    g_vals = np.linspace(PARAM_RANGES["gamma"][0],
                         PARAM_RANGES["gamma"][1], 300)
    # R₀ = 1 line
    ax.plot(g_vals, g_vals / ratio, "r--", lw=2,
            label=r"$\mathcal{R}_0 = 1$")
    # Near-threshold band 
    ax.fill_between(g_vals, 0.5*g_vals/ratio, 2.5*g_vals/ratio,
                    color="gold", alpha=0.25,
                    label=r"$\mathcal{R}_0 \in [0.5, 2.5]$")
    ax.set_xlabel(r"$\gamma$ (recovery rate)",   fontsize=12)
    ax.set_ylabel(r"$\tau$ (transmission rate)", fontsize=12)
    ax.set_title(r"(b)  Posterior samples in $(\gamma, \tau)$ space",
                 fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig(out_dir / "r0_distribution.png", dpi=600,
                bbox_inches="tight")
    plt.close()
    print(f"Saved: {out_dir / 'r0_distribution.png'}")


# ══════════════════════════════════════════════════════════════════════════
#  3.  ABM SIMULATIONS
# ══════════════════════════════════════════════════════════════════════════

def run_batch(G, posterior_samples, n_replicates, tmax, n_timepoints):
    """
    Run EoN.fast_SIR for each posterior sample.

    Returns a list of dicts, each containing params and interpolated
    S(t), I(t), R(t) on a fixed time grid.
    """
    t_fixed  = np.linspace(0, tmax, n_timepoints)
    all_sims = []

    for tau, gamma, rho in tqdm(posterior_samples, desc="Simulations"):
        for rep in range(n_replicates):
            t, S, I, R = EoN.fast_SIR(G, tau, gamma, rho=rho, tmax=tmax)
            all_sims.append({
                'params': {
                    'tau'  : float(tau),
                    'gamma': float(gamma),
                    'rho'  : float(rho),
                },
                'output': {
                    't': t_fixed,
                    'S': np.interp(t_fixed, t, S),
                    'I': np.interp(t_fixed, t, I),
                    'R': np.interp(t_fixed, t, R),
                },
                'replicate_id': rep,
            })

    print(f"\nGenerated {len(all_sims)} simulations "
          f"({len(posterior_samples)} param sets × {n_replicates} replicates)")
    return all_sims


# ══════════════════════════════════════════════════════════════════════════
#  4.  DATASET CONSTRUCTION
# ══════════════════════════════════════════════════════════════════════════

def build_dataset(all_sims, G, net_stats, m, n_replicates, param_ranges):
    n_tp = len(all_sims[0]['output']['t'])
    return {
        'simulations': all_sims,
        'network': {
            'type' : 'barabasi_albert',
            'N'    : G.number_of_nodes(),
            'm'    : m,
            'ratio': net_stats['ratio'],
            'graph': G,
        },
        'metadata': {
            'n_samples'        : len(all_sims),
            'n_replicates'     : n_replicates,
            'n_timepoints'     : n_tp,
            'param_ranges'     : param_ranges,
            'R0_formula'       : 'R0 = (tau/gamma) * (k2_avg - k_avg)/k_avg',
            'sampling_strategy': 'MCMC-NUTS (TruncatedNormal R0)',
            'sigma_R0'         : SIGMA_R0,
        }
    }


# ══════════════════════════════════════════════════════════════════════════
#  5.  TRAJECTORY PLOTS  (corrected R₀ zone thresholds)
# ══════════════════════════════════════════════════════════════════════════

def summarise_for_plot(dataset):
    sims  = dataset['simulations']
    ratio = dataset['network']['ratio']
    t_fix = sims[0]['output']['t']
    all_I = np.array([s['output']['I'] for s in sims])
    all_S = np.array([s['output']['S'] for s in sims])
    all_R = np.array([s['output']['R'] for s in sims])
    R0_vals = np.array(
        [(s['params']['tau']/s['params']['gamma'])*ratio for s in sims]
    )
    return {
        't'    : t_fix,
        'all_I': all_I, 'all_S': all_S, 'all_R': all_R,
        'R0'   : R0_vals,
    }


def plot_sir_uncertainty(full_results, out_dir=PLOTS_DIR):
    """
    Three-panel epidemic trajectory plot.

    R₀ zone thresholds corrected to match the TruncatedNormal(1, 0.7)
    design: the sampler concentrates draws in [0.3, 1.7] approximately,
    so the old thresholds (< 0.5, 0.5–2, > 2) were wrong.
    """
    t       = full_results['t']
    all_I   = full_results['all_I']
    R0_vals = full_results['R0']
    total   = len(R0_vals)

    # ── Corrected thresholds ──────────────────────────────────────────────
    sub_mask  = R0_vals < 0.5                           # sub-critical
    near_mask = (R0_vals >= 0.5) & (R0_vals <= 2.5)    # near threshold ±20%
    out_mask  = R0_vals > 2.5                           # super-critical

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Epidemic Trajectories by R₀ Regime  "
                 r"(TruncatedNormal sampling, $\sigma=0.7$)",
                 fontsize=13, fontweight="bold")

    for ax, mask, label, colour in [
        (axes[0], sub_mask,
         r"Sub-critical  ($\mathcal{R}_0 < 0.5$)",    "steelblue"),
        (axes[1], near_mask,
         r"Near-threshold  ($\mathcal{R}_0 \in [0.5,2.5]$)", "goldenrod"),
        (axes[2], out_mask,
         r"Super-critical  ($\mathcal{R}_0 > 2.5$)",  "firebrick"),
    ]:
        subset = all_I[mask]
        n      = mask.sum()

        if n == 0:
            ax.set_title(f"{label}\n(no trajectories)")
            continue

        mean_I = subset.mean(axis=0)
        p10    = np.percentile(subset, 10, axis=0)
        p25    = np.percentile(subset, 25, axis=0)
        p75    = np.percentile(subset, 75, axis=0)
        p90    = np.percentile(subset, 90, axis=0)

        ax.fill_between(t, p10, p90, color=colour, alpha=0.15,
                        label="10–90th pct")
        ax.fill_between(t, p25, p75, color=colour, alpha=0.30,
                        label="25–75th pct")
        ax.plot(t, mean_I, color=colour, lw=2.0, label="Mean")
        ax.set_title(f"{label}\nn={n}  ({n/total*100:.1f}%)", fontsize=11)
        ax.set_xlabel("Time (days)",          fontsize=11)
        ax.set_ylabel("Number infectious",    fontsize=11)
        ax.set_ylim(bottom=0)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, linestyle="--")
        ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    out = out_dir / "trajectories_by_r0_zone.png"
    plt.savefig(out, dpi=600, bbox_inches="tight")
    plt.close()
    print(f"Saved: {out}")


# ══════════════════════════════════════════════════════════════════════════
#  6.  PARAMETER DISTRIBUTION PLOTS
# ══════════════════════════════════════════════════════════════════════════

def plot_parameter_distributions(data, out_dir=PLOTS_DIR):
    """
    Histogram grid for all sampled parameters and key outcomes.
    Used to verify the R₀ distribution is now near-normal.
    """
    params = {
        'R0'         : (r"$\mathcal{R}_0$",              "blue"),
        'tau'        : (r"$\tau$ (transmission rate)",    "red"),
        'gamma'      : (r"$\gamma$ (recovery rate)",      "green"),
        'rho'        : (r"$\rho$ (initial infected frac)","purple"),
        'peak_I'     : ("Peak infected",                  "orange"),
        'attack_rate': ("Attack rate",                    "gray"),
    }

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    for ax, (col, (label, colour)) in zip(axes, params.items()):
        vals = data[col].dropna()
        ax.hist(vals, bins=50, color=colour, alpha=0.80, edgecolor="white")
        ax.axvline(vals.mean(), color="black", lw=1.8, linestyle="--",
                   label=f"Mean = {vals.mean():.4f}")
        ax.set_title(f"Distribution of {label}",
                     fontsize=11, fontweight="bold")
        ax.set_xlabel(label, fontsize=10)
        ax.set_ylabel("Count",  fontsize=10)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3, axis="y")
        ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle("MCMC Posterior — Parameter & Outcome Distributions\n"
                 r"$\mathcal{R}_0$ should be approximately normal, "
                 r"centred at 1  ($\sigma=0.7$)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    out = out_dir / "parameter_distributions.png"
    plt.savefig(out, dpi=600, bbox_inches="tight")
    plt.close()
    print(f"Saved: {out}")


# ══════════════════════════════════════════════════════════════════════════
#  7.  SAVE DATASET
# ══════════════════════════════════════════════════════════════════════════

def save_dataset(dataset, filepath):
    with open(filepath, "wb") as f:
        pickle.dump(dataset, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Dataset saved: {filepath}  ({len(dataset['simulations'])} sims)")


def save_csv(dataset, filepath):
    sims  = dataset["simulations"]
    ratio = dataset["network"]["ratio"]
    N_pop = dataset["network"]["N"]

    fields = ["sim_id", "replicate_id",
              "tau", "gamma", "rho",
              "R0", "peak_I", "peak_time",
              "final_R", "attack_rate", "near_threshold"]

    with open(filepath, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for sim_id, sim in enumerate(sims):
            tau   = sim["params"]["tau"]
            gamma = sim["params"]["gamma"]
            rho   = sim["params"]["rho"]
            R0    = (tau / gamma) * ratio
            I     = sim["output"]["I"]
            R     = sim["output"]["R"]
            t     = sim["output"]["t"]
            peak_I   = float(I.max())
            peak_time = float(t[I.argmax()]) if peak_I > 0 else float("nan")
            writer.writerow({
                "sim_id"        : sim_id,
                "replicate_id"  : sim.get("replicate_id", 0),
                "tau"           : tau,
                "gamma"         : gamma,
                "rho"           : rho,
                "R0"            : round(R0, 4),
                "peak_I"        : peak_I,
                "peak_time"     : peak_time,
                "final_R"       : float(R[-1]),
                "attack_rate"   : float(R[-1] / N_pop),
                "near_threshold": int(0.5 <= R0 <= 2.5),
            })
    print(f"CSV saved: {filepath}  ({len(sims)} rows)")


# ══════════════════════════════════════════════════════════════════════════
#  8.  MAIN
# ══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    t0_total = time.perf_counter()

    # ── MCMC sampling ─────────────────────────────────────────────────────
    t0_mcmc = time.perf_counter()
    posterior_samples, trace_thinned = run_mcmc()
    mcmc_time = time.perf_counter() - t0_mcmc
    print(f"\nMCMC time: {mcmc_time:.1f}s  ({mcmc_time/60:.1f} min)")

    # ── Verify R₀ distribution ────────────────────────────────────────────
    plot_r0_distribution(posterior_samples)

    # ── Build network ─────────────────────────────────────────────────────
    print(f"\nBuilding BA network (N={N}, m={m}) ...")
    G = nx.barabasi_albert_graph(N, m, seed=42)  # fixed seed for reproducibility
    degrees = np.array([d for _, d in G.degree()])
    
    # ── Run ABM simulations ───────────────────────────────────────────────
    t0_sim = time.perf_counter()
    all_sims = run_batch(G, posterior_samples,
                          n_replicates, tmax, n_timepoints)
    sim_time = time.perf_counter() - t0_sim
    print(f"Simulation time: {sim_time:.1f}s  ({sim_time/60:.1f} min)")

    # ── Build and save dataset ────────────────────────────────────────────
    dataset = build_dataset(all_sims, G, net_stats, m,
                                  n_replicates, PARAM_RANGES)
    full_results = summarise_for_plot(dataset)

    save_dataset(dataset, OUTPUT_PKL)
    save_csv(dataset, OUTPUT_CSV)

    # ── Plots ─────────────────────────────────────────────────────────────
    plot_sir_uncertainty(full_results)

    data = pd.read_csv(OUTPUT_CSV)
    plot_parameter_distributions(data)

    # ── R₀ zone summary ───────────────────────────────────────────────────
    total = len(data)
    print(f"\nR₀ zone breakdown:")
    print(f"  Sub-critical   (R₀ < 0.5)      : "
          f"{(data['R0']<0.5).sum():5d}  "
          f"({(data['R0']<0.5).mean()*100:.1f}%)")
    print(f"  Near-threshold (0.5 ≤ R₀ ≤ 2.5): "
          f"{data['near_threshold'].sum():5d}  "
          f"({data['near_threshold'].mean()*100:.1f}%)")
    print(f"  Super-critical (R₀ > 2.5)       : "
          f"{(data['R0']>2.5).sum():5d}  "
          f"({(data['R0']>2.5).mean()*100:.1f}%)")
    print(f"\n  R₀ mean   : {data['R0'].mean():.4f}")
    print(f"  R₀ median : {data['R0'].median():.4f}")
    print(f"  R₀ std    : {data['R0'].std():.4f}")

    # ── Timing report ─────────────────────────────────────────────────────
    total_time = time.perf_counter() - t0_total
    with open(TIMING_FILE, "w", encoding="utf-8") as f:
        f.write("=" * 55 + "\n")
        f.write("MCMC Sampling — Timing Report\n")
        f.write(f"Run date  : "
                f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
        f.write("=" * 55 + "\n\n")
        f.write(f"Model          : TruncatedNormal R₀ (σ={SIGMA_R0})\n")
        f.write(f"Samples        : {len(posterior_samples)}\n")
        f.write(f"Replicates     : {n_replicates}\n")
        f.write(f"Network size N : {N}\n\n")
        f.write(f"MCMC time      : {mcmc_time:.1f} s "
                f"({mcmc_time/60:.2f} min)\n")
        f.write(f"Simulation time: {sim_time:.1f} s "
                f"({sim_time/60:.2f} min)\n")
        f.write(f"Total time     : {total_time:.1f} s "
                f"({total_time/60:.2f} min)\n")
    print(f"\nTiming saved: {TIMING_FILE}")
    print(f"All outputs  → {DATA_DIR}  |  {PLOTS_DIR}")